In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.8 MB/s eta 0:00:00


In [ ]:
import torch

if torch.cuda.is_available():
    print(f"CUDA is available. Device name: {torch.cuda.get_device_name(0)}")
    if "Tesla T4" in torch.cuda.get_device_name(0):
        print("You are using a T4 GPU.")
    else:
        print("You are using a different GPU.")
else:
    print("CUDA is not available. You are likely using a CPU runtime.")

CUDA is available. Device name: Tesla T4
You are using a T4 GPU.


In [ ]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
model = YOLO("yolo11s-cls.pt")  # load a pretrained model (recommended for training)

In [ ]:
results = model.train(data="FinalizedDataset", epochs=150, imgsz=320, batch=-1)

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=FinalizedDataset, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pr

In [ ]:
model = YOLO("./runs/classify/train/weights/best.pt")

In [ ]:
# Validate the model
metrics = model.val()
metrics.top1  # top1 accuracy
metrics.top5  # top5 accuracy

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
train: /content/FinalizedDataset/train... found 5398 images in 6 classes ✅ 
val: /content/FinalizedDataset/val... found 740 images in 6 classes ✅ 
test: /content/FinalizedDataset/test... found 737 images in 6 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2646.3±1370.8 MB/s, size: 85.5 KB)
val: Scanning /content/FinalizedDataset/val... 740 images, 0 corrupt: 100% ━━━━━━━━━━━━ 740/740 980.3Kit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 47/47 13.0it/s 3.6s
                   all      0.977          1
Speed: 0.3ms preprocess, 1.7ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val3


1.0

In [ ]:
metrics = model.val(split="test")
m = metrics.confusion_matrix.matrix  # shape: (6, 6) for 6 classes

class_names = ['binder', 'file', 'headphone', 'mouse', 'mug', 'pen']
num_classes = len(class_names)

for i in range(num_classes) :
    TP = m[i, i]
    FP = m[:, i].sum() - TP
    FN = m[i, :].sum() - TP
    TN = m.sum() - (TP + FP + FN)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = (TP + TN) / m.sum() if m.sum() > 0 else 0.0

    print(f"Class '{class_names[i]}':")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall:    {recall:.3f}")
    print(f"  F1 Score:  {f1:.3f}")
    print(f"  Accuracy:  {accuracy:.3f}\n")

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
train: /content/FinalizedDataset/train... found 5398 images in 6 classes ✅ 
val: /content/FinalizedDataset/val... found 740 images in 6 classes ✅ 
test: /content/FinalizedDataset/test... found 737 images in 6 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 661.8±533.6 MB/s, size: 75.2 KB)
test: Scanning /content/FinalizedDataset/test... 737 images, 0 corrupt: 100% ━━━━━━━━━━━━ 737/737 5.9Kit/s 0.1s
test: New cache created: /content/FinalizedDataset/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 47/47 8.2it/s 5.7s
                   all      0.961          1
Speed: 0.5ms preprocess, 1.9ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val2
Class 'binder':
  Precision: 0.981
  Recall:    0.972
  F1 Score:  0.977
  Accuracy:  0.993

Class 'file':
  Precision: 0.991
  Recall:    0.991
  F1 Score:  0.991
  Accuracy:  0.997



In [ ]:
# Export the model
model.export(format="tfjs")
model.export(format="onnx")

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/runs/classify/train/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 6) (10.5 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx>=1.12.0', 'onnx2tf>=1.26.3', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...

requirements: AutoUpdate success ✅ 11.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.19.1 opset 22...
ONNX: slimming with onnxslim 0.1.72...
ONNX: export success ✅ 1.0s, saved as '/content/runs/classify/train/weights/best.onnx' (20.8 MB)
Unzipping calibratio


TensorFlow.js: starting export with tensorflowjs 4.22.0...

TensorFlow.js: output node names: Identity:0
TensorFlow.js: running 'tensorflowjs_converter --input_format=tf_frozen_model  --output_node_names=Identity:0 "/content/runs/classify/train/weights/best.pb" "/content/runs/classify/train/weights/best_web_model"'
TensorFlow.js: export success ✅ 9.9s, saved as '/content/runs/classify/train/weights/best_web_model' (21.1 MB)

Export complete (45.9s)
Results saved to /content/runs/classify/train/weights
Predict:         yolo predict task=classify model=/content/runs/classify/train/weights/best_web_model imgsz=320  
Validate:        yolo val task=classify model=/content/runs/classify/train/weights/best_web_model imgsz=320 data=FinalizedDataset  
Visualize:       https://netron.app
Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)

PyTorch: starting from '/content/runs/classify/train/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output sha

'/content/runs/classify/train/weights/best.onnx'